In [1]:
# Generating ground truth data 

In [1]:
from ingest import load_faq_data

documents = load_faq_data()
documents[10]

{'id': '316180784f',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: How many hours per week am I expected to spend on this course?',
 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.'}

In [2]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

113

In [3]:
documents = documents_llm

In [4]:
doc = documents[0]

print(doc["id"]) # we need to have an id, to check if we do good retrieval 
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [5]:
from pydantic import BaseModel

class Questions(BaseModel): # forced output 
    questions: list[str]

In [6]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [7]:
import os 
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [8]:
import json

user_prompt = json.dumps(doc)

In [9]:
doc

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [10]:
user_prompt

'{"id": "74eb249bbf", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "I just discovered the course. Can I still join?", "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."}'

In [11]:
messages = [
    {"role": "system", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [12]:
response = openai_client.responses.parse(
    model="meta-llama/llama-4-scout-17b-16e-instruct",
    input=messages, # we want to enforce a specific strucutre 
    text_format=Questions # structure 
)

In [13]:
result = response.output_parsed
print(result)

questions=['Can I join the course llm-zoomcamp if I just found out about it?', "Is it still possible to join the course if I'm late to the start?", 'If I discover the course late, can I still participate?', 'Can I still join llm-zoomcamp and get a certificate?', 'How late can I join the course and still get a certificate?']


In [14]:
print(result.questions)

['Can I join the course llm-zoomcamp if I just found out about it?', "Is it still possible to join the course if I'm late to the start?", 'If I discover the course late, can I still participate?', 'Can I still join llm-zoomcamp and get a certificate?', 'How late can I join the course and still get a certificate?']


In [15]:
# extract questions from all docs 
from evaluation_utils import llm_structured, calc_price

result, usage = llm_structured(openai_client, data_gen_instructions, user_prompt, Questions)
print(result.questions)

['Can I join the course llm-zoomcamp if I just found out about it?', "Is it still possible to join the course if I'm late?", 'Do I need to submit a project to get a certificate in llm-zoomcamp?', 'What happens if I join the course late, can I still get a certificate?', 'Can I still get a certificate if I join llm-zoomcamp after it started?']


In [16]:
calc_price(usage)

{'input_cost': 2.794e-05, 'output_cost': 3.468e-05, 'total_cost': 6.262e-05}

In [17]:
result.questions

['Can I join the course llm-zoomcamp if I just found out about it?',
 "Is it still possible to join the course if I'm late?",
 'Do I need to submit a project to get a certificate in llm-zoomcamp?',
 'What happens if I join the course late, can I still get a certificate?',
 'Can I still get a certificate if I join llm-zoomcamp after it started?']

In [18]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'Can I join the course llm-zoomcamp if I just found out about it?',
  'document': '74eb249bbf'},
 {'question': "Is it still possible to join the course if I'm late?",
  'document': '74eb249bbf'},
 {'question': 'Do I need to submit a project to get a certificate in llm-zoomcamp?',
  'document': '74eb249bbf'},
 {'question': 'What happens if I join the course late, can I still get a certificate?',
  'document': '74eb249bbf'},
 {'question': 'Can I still get a certificate if I join llm-zoomcamp after it started?',
  'document': '74eb249bbf'}]

In [19]:
import pandas as pd

records_df = pd.DataFrame(records)
records_df

,question,document
0,Can I join the course llm-zoomcamp if I just f...,74eb249bbf
1,Is it still possible to join the course if I'm...,74eb249bbf
2,Do I need to submit a project to get a certifi...,74eb249bbf
3,"What happens if I join the course late, can I ...",74eb249bbf
4,Can I still get a certificate if I join llm-zo...,74eb249bbf


For each document, we:

- convert the document to JSON so we can send it to the LLM
- ask the LLM to return a Questions object
- create one ground truth record for each generated question

In [20]:
# we do all the above on all documents 
from evaluation_utils import llm_structured_retry

def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [22]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents): # first first documents 
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/113 [00:00<?, ?it/s]

In [27]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

0.008402059999999994

In [29]:
# create a dataframe (records)
df_ground_truth = pd.DataFrame(ground_truth)
df_ground_truth

,question,document
0,Can I join the course llm-zoomcamp if I just f...,74eb249bbf
1,Is it still possible to join the course if I'm...,74eb249bbf
2,"I discovered the course late, can I still part...",74eb249bbf
3,Can I still join llm-zoomcamp and get a certif...,74eb249bbf
4,Is there still time to join the course and sub...,74eb249bbf
...,...,...
560,I'm getting a 401 Client Error when trying to ...,4b30b918bc
561,Why does pip install requests library version ...,4b30b918bc
562,How can I install a specific version of the re...,4b30b918bc
563,What if my key is correct but I still get a 40...,4b30b918bc


In [31]:
os.mkdir("data")
df_ground_truth.to_csv("data/ground_truth-new.csv", index=False)